In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## 1.没有记忆时

In [2]:
from langchain.agents import create_agent

agent = create_agent(model="deepseek-chat")

In [4]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我是Orien，我爱吃鸡翅")]}
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='a799c869-c71d-48af-92d8-9cd602cf7986'), AIMessage(content='你好，Orien！很高兴认识你，也欢迎你和我分享关于鸡翅的爱好～  \n鸡翅确实是超棒的美食！无论是香辣烤鸡翅、蒜香蜜汁炸鸡翅，还是可乐鸡翅、泰式甜辣鸡翅……光是想想就让人流口水了！🤤  \n\n你最喜欢哪种做法的鸡翅？或者有没有私藏的烹饪小技巧/宝藏店铺推荐？作为鸡翅同好，我准备好记笔记了！😄🍗', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 14, 'total_tokens': 116, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ad07ce82-4cbd-485c-a148-eebf07547151', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7d8-634b-70c0-9f87-b980c399762c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 

In [6]:
# 第二次调用，询问信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢吃什么？")]}
)

print(response)

{'messages': [HumanMessage(content='我最喜欢吃什么？', additional_kwargs={}, response_metadata={}, id='c0beb39a-6d3e-4675-a3ef-07a64df11805'), AIMessage(content='哈哈，这个问题可难倒我了！虽然我无法知道你具体喜欢什么食物，但可以给你几个小线索自己推理：  \n1. **回忆高频词**：你最近聊天时是否总提到某种食物？比如“火锅”“披萨”或“冰淇淋”？  \n2. **情境联想**：开心时想吃炸鸡？焦虑时爱啃巧克力？疲惫时离不开奶茶？  \n3. **隐藏设定**：如果这是谜题，或许答案藏在你之前的对话里（可惜我看不到历史记录哦～）。  \n\n不如直接告诉我？我还能帮你推荐菜谱或餐厅！ 😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 8, 'total_tokens': 134, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '1bc06263-4385-49aa-84f8-bfc7f0c39fb6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7d9-4200-7e00-bda9-3279cdd4e4be-0', tool_calls=[], i

## 记忆
### Agent的记忆（Memory）分两类
- 短期记忆（short-term-memory）：当前任务或会话的上下文
- 长期记忆（long-term-memory）：跨任务或会话的经验与知识  
区分短期记忆和长期记忆并不是记忆时间的长久，而是记忆的作用域

### 短期记忆
在LangChain短期记忆是通过AgentState实现的，而会话历史（也就是消息列表）是AgentState的一部分
LangChain提供了Checkpointer对象来保存AgentState，每一次用户与AI的交互都会生成一个快照，记录为一个checkpoint。  
同一个会话的多个checkpoint形成一个组，用同一个thread_id来标记

## 2.添加记忆
- 导入并初始化Checkpointer
- 创建Agent，指定Checkpoint
- 调用Agent, 指定thread_id

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()
)

In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我是Orien，我爱吃鸡翅")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='f903a692-56b8-456f-bb3a-4ee6ed0c5d95'), AIMessage(content='你好，Orien！很高兴认识你。鸡翅确实是个很棒的爱好，无论是烤的、炸的、红烧的，还是裹上各种酱汁（比如蜂蜜芥末、甜辣酱、蒜香酱油……），想想都让人流口水。你最喜欢哪种做法的鸡翅？或者有没有自己独特的烹饪秘诀？😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 14, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ec471176-7f3d-4c6e-b4f0-e8b113fe1836', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7f0-5c71-7bc3-8ba0-6d605b8b5c5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 71, 'total_tokens': 85, 'inp

In [10]:
# 第二次调用，询问信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢吃什么？")]},
    config
)

print(response)

{'messages': [HumanMessage(content='你好，我是Orien，我爱吃鸡翅', additional_kwargs={}, response_metadata={}, id='f903a692-56b8-456f-bb3a-4ee6ed0c5d95'), AIMessage(content='你好，Orien！很高兴认识你。鸡翅确实是个很棒的爱好，无论是烤的、炸的、红烧的，还是裹上各种酱汁（比如蜂蜜芥末、甜辣酱、蒜香酱油……），想想都让人流口水。你最喜欢哪种做法的鸡翅？或者有没有自己独特的烹饪秘诀？😄', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 14, 'total_tokens': 85, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 14}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': 'ec471176-7f3d-4c6e-b4f0-e8b113fe1836', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fa7f0-5c71-7bc3-8ba0-6d605b8b5c5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 14, 'output_tokens': 71, 'total_tokens': 85, 'inp